# 01 - Train and monitor

Trains in the background with **Pause / Resume / Stop** buttons and live curves, then evaluates
the held-out TEST block against the baselines. The notebook stays responsive during training (the
buttons only work while no cell is running). Stop ends training after the current batch; the run is
still evaluated, calibrated and saved. Every number here links to a run directory under `runs/`.

In [1]:
# Parameters
CONFIG_PATH = "../configs/default.yaml"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
RUNS_DIR = "../runs"
OVERRIDES = {}            # e.g. {"EPOCHS": 5, "LR": 5e-4, "FOLD_INDEX": -2, "BATCH_SIZE": 64}
EPOCHS = None             # None -> Config.EPOCHS
CALIBRATE_LOSS_WEIGHTS = True

In [2]:
from IPython.display import Markdown, display

import neural_trade  # first: on Windows it puts the CUDA DLLs on PATH before TensorFlow loads
from neural_trade.core.config import Config
from neural_trade.data.processor import split_arrays
from neural_trade.evaluation.baselines import BaselineSet
from neural_trade.evaluation.frame import PredictionFrame
from neural_trade.evaluation.report import evaluate
from neural_trade.experiments.run_context import RunContext
from neural_trade.notebook import TrainingSession
from neural_trade.registries.visualizations import Visualizations
from neural_trade.telemetry.epoch_logger import read_metrics
import tensorflow as tf

print("GPU:", tf.config.list_physical_devices("GPU") or "none - training will run on the CPU")
cfg = Config.from_yaml(CONFIG_PATH).override(CSV_PATH=CSV_PATH, **OVERRIDES)
ctx = RunContext.create(cfg, root=RUNS_DIR, tags=["notebook"])
print("run:", ctx.run_dir)

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
run: ..\runs\20260924T114819Z-e23fd9f-dirty-af67ee43


## Train

This cell returns immediately; the dashboard below keeps updating. The next cell waits for the run to finish.

In [3]:
session = TrainingSession(ctx.config, run_context=ctx, epochs=EPOCHS, calibrate=CALIBRATE_LOSS_WEIGHTS)
display(session.widget())
session.start()

In [4]:
result = session.wait()   # blocks until training, evaluation and calibration are done
print(session.status, "-", len(session.history), "epochs")

finished - 20 epochs


Static copy of the training record (readable in the saved notebook without a running kernel):

In [5]:
display(session.history_frame().round(4))
session.curves_figure().show()

,epoch,seconds,loss,val_loss,val_dir_mcc_h1,val_gauss_dir_mcc_h1,val_pit_ks_h1,val_crps_loss,val_nll_loss,nonfinite_grad_steps
0,0,41.5037,7.3094,5.8501,0.0407,-0.0490,0.1579,1.0691,3.1904,0.0
1,1,26.4432,6.7462,5.8904,-0.0025,-0.1412,0.1706,1.0666,3.1600,0.0
2,2,24.2359,6.5751,5.6538,0.0386,-0.1909,0.1711,1.0487,3.0527,0.0
3,3,38.6859,6.4636,5.5128,-0.0339,-0.1457,0.1695,1.0354,2.9560,0.0
4,4,24.8067,6.3814,5.3683,0.0113,-0.1528,0.1393,1.0029,2.8161,0.0
5,5,23.2389,6.3279,5.5673,-0.1317,-0.1765,0.1296,1.0120,2.8517,0.0
6,6,23.5976,6.2589,5.4588,0.0112,-0.1102,0.1419,1.0089,2.8209,0.0
7,7,23.4173,6.1942,5.2377,-0.0605,-0.1491,0.1145,0.9960,2.7656,0.0
8,8,23.5144,6.1530,5.1922,-0.0175,-0.1727,0.1176,0.9902,2.7205,0.0
9,9,23.4604,6.1072,5.1057,-0.0155,-0.0995,0.0930,0.9877,2.6776,0.0


## Evaluate on the TEST block

Baselines are fitted on the train block; the confidence threshold and calibration come from the cal block.

In [6]:
blocks = split_arrays(ctx.config)
baselines = BaselineSet.fit(blocks["train"]["X"], blocks["train"]["y"], blocks["train"]["last_close"],
                            ctx.config.DIR_DEADBAND_BPS)
test = PredictionFrame.from_result(result, "test")
cal = PredictionFrame.from_result(result, "cal")
report = evaluate(test, ctx.config, baselines=baselines, cal_frame=cal, run_id=ctx.run_id)
report.to_json(ctx.path("eval_report_test.json"))
display(Markdown(report.to_markdown(ctx.path("eval_report_test.md"))))

Dataset length after cleaning: 43500


Date range after cleaning: 2025-10-11 02:30:00+00:00 to 2025-11-10 07:29:00+00:00


# Evaluation report - test split - run `20260924T114819Z-e23fd9f-dirty-af67ee43`

n = 7236 samples; direction metrics exclude moves within 5 bps (neutral mask); n_eff counts non-overlapping outcomes.

| metric | h0 | h1 | h2 |
|---|---|---|---|
| n_eff | 723 | 482 | 361 |
| direction MCC | -0.0098 | -0.0130 | -0.0055 |
| direction AUC | 0.4988 | 0.4997 | 0.5073 |
| direction ECE | 0.0276 | 0.0379 | 0.0240 |
| Gaussian MCC | 0.0000 | 0.0000 | 0.0478 |
| Gaussian AUC | 0.5000 | 0.5000 | 0.5187 |
| delta EV | 0.0000 | 0.0000 | 0.0004 |
| delta corr | 0.0000 | 0.0000 | 0.0192 |
| skill vs zero | 0.0000 | 0.0000 | -0.0001 |
| CRPS ($) | 105.8291 | 127.5036 | 145.1798 |
| PIT KS | 0.0392 | 0.0384 | 0.0438 |
| var/err2 Spearman | 0.2414 | 0.2391 | 0.2315 |
| coverage 90% | 0.9027 | 0.9059 | 0.9122 |

## Confidence gap (accuracy of the more confident half minus the less confident half)

| horizon | gap | 95% CI | verdict |
|---|---|---|---|
| h0 | 0.0063 | [-0.0328, 0.0395] | NOISE |
| h1 | 0.0152 | [-0.0252, 0.0567] | NOISE |
| h2 | 0.0071 | [-0.0391, 0.0574] | NOISE |

## Coherence across horizons

- mag_order_full: 1.0000
- unanimity: 0.2872
- delta_dir_align_all: 0.1617
- coherence_primary: 0.6064

## Against baselines (fit on the training block)

| baseline | metric | h0 | h1 | h2 |
|---|---|---|---|---|
| zero_delta | delta/rmse | does not beat | does not beat | does not beat |
| zero_delta | delta/mae | does not beat | does not beat | does not beat |
| zero_delta | delta/ev | does not beat | does not beat | beats |
| zero_delta | delta/corr | does not beat | does not beat | beats |
| zero_delta | delta/skill_vs_zero | does not beat | does not beat | does not beat |
| mean_delta | delta/rmse | beats | beats | beats |
| mean_delta | delta/mae | beats | beats | beats |
| mean_delta | delta/ev | does not beat | does not beat | beats |
| mean_delta | delta/corr | beats | does not beat | beats |
| mean_delta | delta/skill_vs_zero | beats | beats | beats |
| class_prior | direction/mcc | does not beat | does not beat | does not beat |
| class_prior | direction/auc | does not beat | does not beat | beats |
| class_prior | direction/brier | does not beat | does not beat | does not beat |
| class_prior | direction/ece_pos | does not beat | does not beat | does not beat |
| class_prior | direction/acc | beats | beats | beats |
| class_prior | direction/bal_acc | does not beat | does not beat | does not beat |
| const_var | variance/crps | beats | beats | beats |
| const_var | variance/nll | beats | beats | beats |
| const_var | variance/pit_ks | beats | beats | beats |
| const_var | variance/corr_var_err2_spearman | beats | beats | beats |
| logreg_lags | direction/mcc | does not beat | does not beat | does not beat |
| logreg_lags | direction/auc | does not beat | does not beat | does not beat |
| logreg_lags | direction/brier | does not beat | does not beat | does not beat |
| logreg_lags | direction/ece_pos | does not beat | does not beat | does not beat |
| logreg_lags | direction/acc | does not beat | does not beat | does not beat |
| logreg_lags | direction/bal_acc | does not beat | does not beat | does not beat |


In [7]:
Visualizations.build("eval_report", test, ctx.config).show()

## Learned indicator periods

In [8]:
Visualizations.build("indicator_evolution", ctx.path("metrics.jsonl"), ctx.config).show()